<a href="https://colab.research.google.com/github/bhumikasingh11/travel-planner-ai/blob/main/AGENTPLANNER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-core langchain-groq langchain-community langgraph gradio

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=Warning)

In [ ]:
import logging
logging.getLogger('langgraph').setLevel(logging.ERROR)

In [ ]:
import os
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_groq import ChatGroq
from IPython.display import display, Image
import gradio as gr

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


KeyboardInterrupt: 

In [ ]:
# ============================================
# CELL 4: Set up Groq API Key (FIXED LOGIC)
# ============================================

# First, set your hardcoded key as fallback


# Try to get from Colab Secrets (this will override the hardcoded one if exists)
try:
    from google.colab import userdata
    secret_key = userdata.get('GROQ_API_KEY')
    if secret_key:
        GROQ_API_KEY = secret_key
        print("✅ API key loaded from Colab Secrets")
    else:
        print("⚠️ No secret found, using hardcoded key")
except:
    print("⚠️ Colab secrets not available, using hardcoded key")

# Verify we have a key
if GROQ_API_KEY:
    print("✅ API key is ready!")
else:
    print("❌ ERROR: No API key found!")

✅ API key loaded from Colab Secrets
✅ API key is ready!


In [ ]:
from langchain_groq import ChatGroq

if GROQ_API_KEY:
    llm = ChatGroq(
        temperature=0,
        groq_api_key=GROQ_API_KEY,
        model_name="llama-3.3-70b-versatile"
    )
    print("✅ LLM initialized successfully!")
else:
    llm = None
    print("❌ No API key found!")

In [ ]:
if llm:
    try:
        test_response = llm.invoke("Say 'API is working!'")
        print("✅ Test successful:", test_response.content)
    except Exception as e:
        print("❌ Test failed:", e)

In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage

class PlannerState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage], "the messages in the conversation"]
    city: str
    interests: List[str]
    itinerary: str

print("✅ PlannerState defined")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

itinerary_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert travel assistant with deep knowledge of destinations worldwide.
    Create a detailed day trip itinerary for {city} based on the user's interests: {interests}.

    Please provide:
    - A warm greeting
    - Morning activities (9 AM - 12 PM)
    - Lunch recommendations
    - Afternoon activities (1 PM - 5 PM)
    - Evening activities (6 PM - 9 PM)
    - Practical tips (transport, weather, local customs)

    Format with emojis and bullet points for easy reading."""),
    ("human", "Create an itinerary for my day trip to {city}."),
])

print("✅ Prompt template defined")

In [ ]:
def input_city(state: PlannerState) -> PlannerState:
    """Node 1: Get city from user"""
    print("\n" + "="*50)
    print("📍 STEP 1: Where would you like to go?")
    print("="*50)
    user_message = input("Enter city name: ")
    return {
        **state,
        "city": user_message,
        "messages": state['messages'] + [HumanMessage(content=user_message)]
    }

def input_interest(state: PlannerState) -> PlannerState:
    """Node 2: Get interests from user"""
    print("\n" + "="*50)
    print(f"🎯 STEP 2: What are you interested in {state['city']}?")
    print("="*50)
    print("Examples: history, food, nature, museums, shopping, adventure, culture")
    user_message = input("Enter your interests (comma-separated): ")
    return {
        **state,
        "interests": [interest.strip() for interest in user_message.split(",")],
        "messages": state['messages'] + [HumanMessage(content=user_message)]
    }

def create_itinerary(state: PlannerState) -> PlannerState:
    """Node 3: Generate the itinerary using LLM"""
    print("\n" + "="*50)
    print(f"✈️ STEP 3: Creating your personalized itinerary for {state['city']}...")
    print("="*50)

    response = llm.invoke(
        itinerary_prompt.format_messages(
            city=state['city'],
            interests=', '.join(state['interests'])
        )
    )

    print("\n" + "🎉"*20)
    print("YOUR PERSONALIZED ITINERARY")
    print("🎉"*20)
    print(response.content)
    print("\n" + "="*50)
    print("✅ Enjoy your trip! 🗺️")
    print("="*50)

    return {
        **state,
        "messages": state['messages'] + [AIMessage(content=response.content)],
        "itinerary": response.content,
    }

print("✅ Node functions defined")

In [ ]:
from langgraph.graph import StateGraph, END

# Create the workflow
workflow = StateGraph(PlannerState)

# Add nodes
workflow.add_node("input_city", input_city)
workflow.add_node("input_interest", input_interest)
workflow.add_node("create_itinerary", create_itinerary)

# Add edges (the flow)
workflow.set_entry_point("input_city")
workflow.add_edge("input_city", "input_interest")
workflow.add_edge("input_interest", "create_itinerary")
workflow.add_edge("create_itinerary", END)

# Compile the app
app = workflow.compile()

print("✅ Multi-Agent Graph compiled successfully!")
print("\n📊 Graph Structure:")
print("input_city → input_interest → create_itinerary → END")

In [ ]:
# ============================================
# CELL 11: Display Graph Visualization (Optional)
# ============================================
from IPython.display import display, Image
from langchain_core.runnables.graph import MermaidDrawMethod

try:
    print("Generating graph visualization...")
    display(
        Image(
            app.get_graph().draw_mermaid_png(
                draw_method=MermaidDrawMethod.API
            )
        )
    )
    print("✅ Graph visualization displayed above")
except Exception as e:
    print(f"⚠️ Could not display graph: {e}")
    print("This is optional - the planner will still work!")

In [ ]:
def run_travel_planner():
    """Main function to run the travel planner"""
    print("\n" + "🌍"*20)
    print("WELCOME TO THE AI TRAVEL PLANNER")
    print("🌍"*20)
    print("\nI'll help you plan the perfect day trip!\n")

    # Initialize state
    initial_state = {
        "messages": [HumanMessage(content="I want to plan a day trip")],
        "city": "",
        "interests": [],
        "itinerary": "",
    }

    # Run the graph
    final_state = None
    for output in app.stream(initial_state):
        final_state = output

    return final_state

# RUN THE PLANNER - Uncomment the line below to run:
run_travel_planner()

In [ ]:
def quick_itinerary(city: str, interests: str) -> str:
    """
    Quick function to get itinerary without the interactive graph.
    Use this for direct calls or API integration.

    Example:
        result = quick_itinerary("Paris", "museums, food, eiffel tower")
        print(result)
    """
    interests_list = [i.strip() for i in interests.split(",")]
    response = llm.invoke(
        itinerary_prompt.format_messages(
            city=city,
            interests=', '.join(interests_list)
        )
    )
    return response.content

In [ ]:
print("="*50)
print("📚 EXAMPLE ITINERARIES")
print("="*50)

examples = [
    ("Paris", "Eiffel Tower, Louvre museum, French cuisine, Seine cruise"),
    ("Tokyo", "sushi, temples, shopping, anime culture"),
    ("Rome", "Colosseum, pasta, Vatican, gelato"),
]

for city, interests in examples:
    print(f"\n📍 {city}: {interests}")
    print("-"*40)
    try:
        result = quick_itinerary(city, interests)
        print(result[:200] + "...\n")  # Show first 200 chars
    except Exception as e:
        print(f"Error: {e}")

print("\n💡 To generate a full itinerary, use: quick_itinerary('Your City', 'your interests')")

In [ ]:
def save_itinerary(city: str, interests: str, filename: str = None):
    """Generate and save itinerary to a text file"""

    if filename is None:
        filename = f"itinerary_{city.lower().replace(' ', '_')}.txt"

    itinerary = quick_itinerary(city, interests)

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"TRAVEL ITINERARY\n")
        f.write(f"="*50 + "\n")
        f.write(f"Destination: {city}\n")
        f.write(f"Interests: {interests}\n")
        f.write(f"="*50 + "\n\n")
        f.write(itinerary)

    print(f"✅ Itinerary saved to: {filename}")
    return filename

# Example: Save an itinerary (uncomment to use)
# save_itinerary("Bangkok", "temples, street food, night markets")

In [ ]:
def find_travel_twin():
    """Interactive quiz to find user's travel style"""

    print("\n" + "🎭"*20)
    print("TRIP TWIN - FIND YOUR TRAVEL PERSONALITY")
    print("🎭"*20)

    questions = [
        ("What excites you most about travel?",
         ["a) History & Culture", "b) Nature & Adventure", "c) Food & Cuisine", "d) Relaxation & Luxury", "e) Photography & Sightseeing"]),

        ("Your ideal vacation pace?",
         ["a) Packed schedule, see everything!", "b) Balanced mix of activities", "c) Spontaneous and flexible", "d) Slow and relaxed"]),

        ("What's your travel budget preference?",
         ["a) Budget backpacker", "b) Mid-range comfortable", "c) Luxury splurge", "d) Whatever it takes for unique experiences"])
    ]

    scores = {"culture": 0, "adventure": 0, "foodie": 0, "luxury": 0, "photo": 0}

    for q_num, (question, options) in enumerate(questions, 1):
        print(f"\n📌 Question {q_num}: {question}")
        for i, opt in enumerate(options, 1):
            print(f"   {i}. {opt}")

        while True:
            try:
                choice = int(input("\nYour choice (1-{}): ".format(len(options))))
                if 1 <= choice <= len(options):
                    break
            except:
                pass
            print("Invalid choice. Try again.")

        # Map answers to travel styles
        if q_num == 1:
            if choice == 1: scores["culture"] += 3
            elif choice == 2: scores["adventure"] += 3
            elif choice == 3: scores["foodie"] += 3
            elif choice == 4: scores["luxury"] += 3
            elif choice == 5: scores["photo"] += 3
        elif q_num == 2:
            if choice == 1: scores["adventure"] += 2
            elif choice == 2:
                for k in scores: scores[k] += 1
            elif choice == 3: scores["adventure"] += 1
            elif choice == 4: scores["luxury"] += 2
        elif q_num == 3:
            if choice == 1: pass  # budget doesn't affect style
            elif choice == 2: scores["culture"] += 1
            elif choice == 3: scores["luxury"] += 2
            elif choice == 4: scores["adventure"] += 1

    # Determine primary travel style
    travel_style = max(scores, key=scores.get)

    style_descriptions = {
        "culture": ("🏛️ Culture Buff", "Museums, historical sites, local traditions, art galleries"),
        "adventure": ("⚡ Adventure Seeker", "Hiking, extreme sports, unique experiences, off-the-beaten-path"),
        "foodie": ("🍜 Foodie Explorer", "Street food, cooking classes, local markets, fine dining"),
        "luxury": ("💎 Luxury Traveler", "5-star hotels, fine dining, premium experiences, spas"),
        "photo": ("📸 Instagram Hunter", "Viewpoints, golden hour spots, colorful streets, drone shots")
    }

    style_name, style_description = style_descriptions[travel_style]

    print("\n" + "="*50)
    print(f"✨ YOUR TRIP TWIN IS: {style_name} ✨")
    print("="*50)
    print(f"\n{style_description}")
    print("\n💡 Your personalized recommendations will focus on:", style_description.split(",")[0])

    return travel_style, style_name



In [ ]:
def get_exchange_rate_to_inr():
    """Get approximate exchange rate (simplified - uses common rates)"""
    # Approximate rates as of 2026
    rates = {
        "USD": 86.50,
        "EUR": 94.20,
        "GBP": 109.80,
        "JPY": 0.57,
        "INR": 1.00,
        "AED": 23.55,
        "SGD": 64.30,
        "AUD": 54.90,
        "CAD": 60.75
    }
    return rates

def calculate_budget_inr(city: str, duration_days: int = 1, hotel_style: str = "mid-range"):
    """Calculate estimated trip budget in Indian Rupees"""

    hotel_map = {
        "budget": "1-2 stars",
        "mid-range": "3 stars",
        "luxury": "4-5 stars"
    }

    hotel_type = hotel_map.get(hotel_style, "3 stars")

    budget_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a travel budget expert. Calculate estimated daily costs in {city} for a {hotel_type} hotel.

        Return ONLY a JSON with this exact structure (all costs in local currency):
        {{
            "attractions": {{"cost": number, "currency": "USD/EUR/GBP/etc", "details": "main attractions cost"}},
            "food": {{"cost": number, "currency": "USD/EUR/GBP/etc", "details": "breakfast+lunch+dinner"}},
            "transport": {{"cost": number, "currency": "USD/EUR/GBP/etc", "details": "local transport daily"}},
            "hotel": {{"cost": number, "currency": "USD/EUR/GBP/etc", "details": "per night cost"}},
            "total": number,
            "saving_tip": "one practical money-saving tip"
        }}

        Be realistic and reasonable. For cheaper cities like India/SE Asia, costs should be lower.
        For expensive cities like Tokyo/London/New York, costs should be higher.
        """),
        ("human", f"Calculate 1-day budget for {city}, {hotel_type} hotel. Be specific with numbers.")
    ])

    response = llm.invoke(budget_prompt.format_messages(city=city, hotel_type=hotel_type))

    import json
    import re

    try:
        # Try to extract JSON from response
        json_match = re.search(r'\{.*\}', response.content, re.DOTALL)
        if json_match:
            budget = json.loads(json_match.group())
        else:
            budget = json.loads(response.content)

        # Get exchange rates
        rates = get_exchange_rate_to_inr()
        currency = budget.get('attractions', {}).get('currency', 'USD')

        # Convert to INR
        exchange_rate = rates.get(currency, 86.50)

        # Convert all costs to INR
        for category in ['attractions', 'food', 'transport', 'hotel']:
            if category in budget and 'cost' in budget[category]:
                cost_local = budget[category]['cost']
                cost_inr = cost_local * exchange_rate
                budget[category]['cost_inr'] = round(cost_inr, 2)
                budget[category]['currency_inr'] = 'INR'

        if 'total' in budget:
            budget['total_inr'] = round(budget['total'] * exchange_rate, 2)

        budget['exchange_rate'] = exchange_rate
        budget['original_currency'] = currency

        return budget

    except Exception as e:
        # Fallback budget for common cities
        return create_fallback_budget(city, hotel_style)

def create_fallback_budget(city: str, hotel_style: str):
    """Create realistic fallback budget if API fails"""

    city_lower = city.lower()

    # Define city tiers (daily costs in USD)
    if any(c in city_lower for c in ['tokyo', 'london', 'new york', 'paris', 'singapore', 'zurich']):
        tier = "expensive"
        base_costs = {"attractions": 50, "food": 60, "transport": 15, "hotel": 150}
    elif any(c in city_lower for c in ['bangkok', 'bali', 'hanoi', 'jaipur', 'kuala lumpur']):
        tier = "cheap"
        base_costs = {"attractions": 15, "food": 15, "transport": 5, "hotel": 35}
    else:
        tier = "medium"
        base_costs = {"attractions": 30, "food": 35, "transport": 10, "hotel": 80}

    # Adjust for hotel style
    if hotel_style == "budget":
        base_costs["hotel"] *= 0.6
    elif hotel_style == "luxury":
        base_costs["hotel"] *= 2.5

    rates = get_exchange_rate_to_inr()
    exchange_rate = rates.get('USD', 86.50)

    budget = {
        "attractions": {"cost": base_costs["attractions"], "currency": "USD", "cost_inr": round(base_costs["attractions"] * exchange_rate, 2), "details": "Main attractions entry fees"},
        "food": {"cost": base_costs["food"], "currency": "USD", "cost_inr": round(base_costs["food"] * exchange_rate, 2), "details": "3 meals at local restaurants"},
        "transport": {"cost": base_costs["transport"], "currency": "USD", "cost_inr": round(base_costs["transport"] * exchange_rate, 2), "details": "Public transport day pass"},
        "hotel": {"cost": base_costs["hotel"], "currency": "USD", "cost_inr": round(base_costs["hotel"] * exchange_rate, 2), "details": f"{hotel_style} hotel per night"},
        "total": sum(base_costs.values()),
        "total_inr": round(sum(base_costs.values()) * exchange_rate, 2),
        "saving_tip": "Book attractions online in advance for discounts!",
        "exchange_rate": exchange_rate,
        "original_currency": "USD"
    }

    return budget

In [ ]:
def enhanced_trip_planner():
    """Complete trip planner with Trip Twin + Budget Calculator"""

    print("\n" + "🌟"*20)
    print("WELCOME TO THE ULTIMATE TRIP PLANNER")
    print("🌟"*20)

    # STEP 1: Find Travel Twin
    travel_style, style_name = find_travel_twin()

    # STEP 2: Get destination
    print("\n" + "="*50)
    city = input("\n📍 Where would you like to travel? ").strip()

    # STEP 3: Get interests (enhanced with travel style)
    print(f"\n🎯 Based on your {style_name} personality...")

    style_interests = {
        "culture": "historical sites, museums, local traditions, architecture",
        "adventure": "hiking, outdoor activities, unique experiences, thrill seeking",
        "foodie": "local cuisine, street food, cooking classes, food markets",
        "luxury": "fine dining, premium experiences, spas, shopping",
        "photo": "scenic viewpoints, golden hour spots, colorful locations"
    }

    suggested = style_interests.get(travel_style, "local experiences")
    print(f"💡 Suggested interests for you: {suggested}")

    interests = input("\nEnter your interests (comma-separated): ").strip()
    if not interests:
        interests = suggested

    # STEP 4: Choose budget level
    print("\n💰 Select your budget preference:")
    print("   1. Budget (Backpacker style)")
    print("   2. Mid-range (Comfortable)")
    print("   3. Luxury (Premium experience)")

    budget_choice = input("\nYour choice (1-3): ").strip()
    budget_map = {"1": "budget", "2": "mid-range", "3": "luxury"}
    hotel_style = budget_map.get(budget_choice, "mid-range")

    # STEP 5: Generate personalized itinerary prompt with travel style
    enhanced_prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are a travel assistant specializing in {style_name} travel experiences.

        Create a DETAILED day trip itinerary for {{city}} based on:
        - Travel style: {style_name}
        - Interests: {{interests}}
        - Budget level: {hotel_style}

        Include:
        ✨ 8:00 AM - 9:00 PM schedule
        🎯 Hidden gems and local favorites
        📍 Must-visit spots matching their style
        🍽️ Meal recommendations with price estimates
        💡 Insider tips

        Make it EXCITING and PERSONALIZED for their travel personality!
        """),
        ("human", "Create my personalized {style_name} itinerary for {city}!")
    ])

    print("\n" + "✈️"*20)
    print("GENERATING YOUR PERSONALIZED ITINERARY...")
    print("✈️"*20 + "\n")

    response = llm.invoke(
        enhanced_prompt.format_messages(
            city=city,
            interests=interests,
            style_name=style_name
        )
    )

    print(response.content)

    # STEP 6: Show budget in Rupees
    print("\n" + "💰"*20)
    print("SMART BUDGET CALCULATOR (IN RUPEES)")
    print("💰"*20)

    budget = calculate_budget_inr(city, 1, hotel_style)

    print(f"\n📍 Destination: {city.upper()}")
    print(f"🏨 Hotel Style: {hotel_style.upper()}")
    print(f"💱 Exchange Rate: 1 {budget.get('original_currency', 'USD')} = ₹{budget.get('exchange_rate', 86.50)}")

    print("\n" + "─"*40)
    print("📊 DAILY COST BREAKDOWN")
    print("─"*40)

    if 'attractions' in budget:
        print(f"🎟️ ATTRACTIONS:  ₹{budget['attractions'].get('cost_inr', 0):,.2f}")
        print(f"   └─ {budget['attractions'].get('details', 'Entry fees')}")

    if 'food' in budget:
        print(f"🍽️ FOOD:         ₹{budget['food'].get('cost_inr', 0):,.2f}")
        print(f"   └─ {budget['food'].get('details', '3 meals')}")

    if 'transport' in budget:
        print(f"🚇 TRANSPORT:    ₹{budget['transport'].get('cost_inr', 0):,.2f}")
        print(f"   └─ {budget['transport'].get('details', 'Local travel')}")

    if 'hotel' in budget:
        print(f"🏨 HOTEL:        ₹{budget['hotel'].get('cost_inr', 0):,.2f}")
        print(f"   └─ {budget['hotel'].get('details', 'Per night')}")

    print("─"*40)
    print(f"💰 TOTAL (1 DAY): ₹{budget.get('total_inr', 0):,.2f}")
    print("─"*40)

    print(f"\n💡 SAVINGS TIP: {budget.get('saving_tip', 'Book in advance for best deals!')}")

    # STEP 7: Multi-day calculation (optional)
    print("\n" + "─"*40)
    days = input("\n📅 How many days are you planning to stay? (Press Enter for 1 day): ").strip()
    if days and days.isdigit():
        days = int(days)
        total_cost = budget.get('total_inr', 0) * days
        print(f"\n💰 TOTAL FOR {days} DAYS: ₹{total_cost:,.2f}")

        # Add flight estimate (simplified)
        if days > 1:
            flight_estimate = 25000 if "india" not in city.lower() else 8000
            print(f"✈️ ESTIMATED FLIGHT: ₹{flight_estimate:,.2f} (round trip)")
            print(f"🎯 GRAND TOTAL (incl. flight): ₹{total_cost + flight_estimate:,.2f}")

    print("\n" + "🎉"*20)
    print("HAPPY TRAVELS! 🗺️✨")
    print("🎉"*20)

    return {
        "city": city,
        "travel_style": travel_style,
        "interests": interests,
        "budget_level": hotel_style,
        "itinerary": response.content,
        "budget": budget
    }

In [ ]:
result = enhanced_trip_planner()


In [ ]:
def quick_enhanced_plan(city: str, interests: str, travel_style: str = "culture", hotel_style: str = "mid-range"):
    """Quick version without interactive questions"""

    enhanced_prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are a {travel_style} travel expert. Create an exciting day itinerary for {{city}} based on interests: {{interests}}"),
        ("human", "Create my itinerary!")
    ])

    itinerary = llm.invoke(enhanced_prompt.format_messages(city=city, interests=interests))
    budget = calculate_budget_inr(city, 1, hotel_style)

    return {
        "itinerary": itinerary.content,
        "budget": budget
    }

# Example usage:
# result = quick_enhanced_plan("Bangkok", "temples, street food", "foodie", "budget")
# print(result["itinerary"])
# print(f"Total: ₹{result['budget']['total_inr']:,.2f}")

In [ ]:

def save_complete_plan(result):
    """Save the entire trip plan to a file"""

    filename = f"trip_plan_{result['city'].lower().replace(' ', '_')}.txt"

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("YOUR PERSONALIZED TRIP PLAN\n")
        f.write("="*60 + "\n\n")

        f.write(f"📍 Destination: {result['city']}\n")
        f.write(f"🎭 Travel Style: {result['travel_style']}\n")
        f.write(f"🎯 Interests: {result['interests']}\n")
        f.write(f"💰 Budget Level: {result['budget_level']}\n\n")

        f.write("="*60 + "\n")
        f.write("ITINERARY\n")
        f.write("="*60 + "\n")
        f.write(result['itinerary'] + "\n\n")

        f.write("="*60 + "\n")
        f.write("BUDGET BREAKDOWN (IN RUPEES)\n")
        f.write("="*60 + "\n")

        budget = result['budget']
        f.write(f"🎟️ Attractions:  ₹{budget.get('attractions', {}).get('cost_inr', 0):,.2f}\n")
        f.write(f"🍽️ Food:         ₹{budget.get('food', {}).get('cost_inr', 0):,.2f}\n")
        f.write(f"🚇 Transport:    ₹{budget.get('transport', {}).get('cost_inr', 0):,.2f}\n")
        f.write(f"🏨 Hotel:        ₹{budget.get('hotel', {}).get('cost_inr', 0):,.2f}\n")
        f.write("─"*40 + "\n")
        f.write(f"💰 TOTAL:        ₹{budget.get('total_inr', 0):,.2f}\n\n")

        f.write(f"💡 Tip: {budget.get('saving_tip', 'Happy travels!')}\n")

    print(f"✅ Trip plan saved to: {filename}")
    return filename

# Save your result
if 'result' in locals():
    save_complete_plan(result)

In [ ]:
# Clean up Streamlit files
!rm -f app.py
!pkill -f streamlit 2>/dev/null
print("✅ Cleaned up Streamlit files")

In [ ]:
# ============================================
# 🌍 ENHANCED GRADIO UI - FULL REPLACEMENT
# ============================================
!pip install -q gradio
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import json, re

# ── API Key ────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except:
    GROQ_API_KEY = "gsk_YOUR_KEY_HERE"
    print("⚠️ Using hardcoded API key")

llm = ChatGroq(temperature=0, groq_api_key=GROQ_API_KEY, model_name="llama-3.3-70b-versatile")
print("✅ LLM Ready!")

# ── Budget Helper ──────────────────────────────────────────
def get_budget_inr(city, budget_level, days):
    city_lower = city.lower()

    # ── City tier ──────────────────────────────────────────
    if any(c in city_lower for c in ['tokyo','london','new york','paris','singapore','dubai','zurich']):
        base = {"budget": 4500, "mid-range": 13000, "luxury": 35000}
    elif any(c in city_lower for c in ['bangkok','bali','hanoi','kuala lumpur','colombo']):
        base = {"budget": 2000, "mid-range": 5500, "luxury": 15000}
    elif any(c in city_lower for c in ['mumbai','delhi','bangalore','bengaluru','goa','kerala',
                                        'rajasthan','jaipur','udaipur','jodhpur','jaisalmer',
                                        'agra','varanasi','kolkata','chennai','hyderabad',
                                        'pune','amritsar','shimla','manali','rishikesh']):
        base = {"budget": 1500, "mid-range": 4000, "luxury": 12000}
    else:
        base = {"budget": 3000, "mid-range": 8000, "luxury": 22000}

    # ── Daily cost (budget_level comes in as "Budget"/"Mid-range"/"Luxury") ──
    daily = base.get(budget_level.lower(), base["mid-range"])

    total = daily * days

    # ── Flight: 0 for any Indian city ─────────────────────
    india_cities = ['india','mumbai','delhi','bangalore','bengaluru','goa','kerala',
                    'rajasthan','jaipur','udaipur','jodhpur','jaisalmer','agra',
                    'varanasi','kolkata','chennai','hyderabad','pune','amritsar',
                    'shimla','manali','rishikesh','mysore','kochi','surat','nagpur']
    is_india = any(c in city_lower for c in india_cities)

    flight = 0 if is_india else (
        18000 if budget_level == "Budget" else
        30000 if budget_level == "Mid-range" else
        65000
    )

    tips = {
        "budget":    "🎒 Stay in hostels, eat at local dhabas/street stalls, use public transit!",
        "mid-range": "🧳 Book hotels 3–4 weeks early, mix street food with sit-down meals.",
        "luxury":    "💎 Ask hotels for airport transfers & skip-the-line passes — worth it!"
    }

    return {
        "daily":       daily,
        "total":       total,
        "flight":      flight,
        "grand_total": total + flight,
        "tip":         tips.get(budget_level.lower(), "Book in advance!")
    }
# ── Itinerary Generator ────────────────────────────────────
def generate_itinerary(city, interests, travel_style, days, budget_level):
    style_prompts = {
        "🏟️ Culture Buff":      "Focus on museums, heritage sites, architecture, and local history.",
        "🏂 Adventure Seeker":  "Focus on hiking, extreme sports, outdoor thrills, and hidden trails.",
        "🍜 Foodie Explorer":   "Focus on street food, local restaurants, food markets, and cooking classes.",
        "💎 Luxury Traveler":   "Focus on 5-star stays, fine dining, spas, and premium curated experiences.",
        "📹 Insta Hunter":      "Focus on scenic viewpoints, golden-hour spots, photogenic streets, and drone-worthy locations."
    }
    style_tip = style_prompts.get(travel_style, "")

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are an enthusiastic expert travel planner.
{style_tip}
Budget level: {budget_level}.

Format your response EXACTLY like this for each day:

---
## 🌅 Day N — [Catchy Day Title]

**Morning (8:00 AM – 12:00 PM)**
- 🕰️ 8:00 AM — [Activity] — 📍 [Place] — 💡 [Quick tip]
- 🕰️ 10:00 AM — [Activity] ...

**Afternoon (12:00 PM – 5:00 PM)**
- 🍽️ 12:30 PM — Lunch at [Place] — ₹[approx cost]
- 🕰️ 2:00 PM — [Activity] ...

**Evening (5:00 PM – 9:00 PM)**
- 🕰️ 5:30 PM — [Activity] ...
- 🌙 8:00 PM — Dinner at [Place] — ₹[approx cost]

🔥 **Hidden Gem:** [One secret spot locals love]
🗯️ **Insider Tip:** [One practical tip for this day]
---

Make it exciting, specific, and full of personality!"""),
        ("human", f"Create my {days}-day {travel_style} itinerary for {city}! My interests: {interests}")
    ])
    response = llm.invoke(prompt.format_messages())
    return response.content

# ── Main Function ──────────────────────────────────────────
def trip_planner(city, interests, travel_style, days, budget_level, progress=gr.Progress()):
    if not city.strip():
        return (
            "### ❌ Please enter a destination city!",
            "", "", ""
        )

    if "," in city:
        return (
            f"### ⚠️ One city at a time please!\n\n"
            f"**You typed:** `{city}`\n\n"
            f"This planner works best with **one destination at a time**.\n\n"
            f"👉 Try running it twice — once for `{city.split(',')[0].strip()}` and once for `{city.split(',')[1].strip()}`!",
            "", "", ""
        )
    progress(0.1, desc="🗺️ Mapping your adventure...")
    itinerary = generate_itinerary(city, interests, travel_style, int(days), budget_level)

    progress(0.85, desc="💰 Crunching rupees...")
    b = get_budget_inr(city, budget_level, int(days))

    progress(0.95, desc="✨ Polishing your plan...")

    # Budget card
    flight_row = f"✈️ **Round-trip flight est.** | ₹{b['flight']:,.0f}" if b['flight'] > 0 else "✈️ **Flight** | Domestic — check MakeMyTrip"
    budget_md = f"""
## 💰 Your Trip Budget

| Category | Per Day | {int(days)} Days |
|---|---|---|
| 🏨 Stay + food + local transport | ₹{b['daily']:,.0f} | ₹{b['total']:,.0f} |
| {flight_row} | — |

### 🧾 Grand Total: ₹{b['grand_total']:,.0f}

> {b['tip']}
"""

    # Destination card
    style_emoji = travel_style.split()[0]
    dest_md = f"""
## {style_emoji} {city} — {travel_style}

**📆 Duration:** {int(days)} {'day' if int(days)==1 else 'days'}
**😎 Vibe:** {interests if interests.strip() else 'Local experiences'}
**💰 Budget tier:** {budget_level}

---
*Scroll down for your full day-by-day plan! 👇*
"""

    tip_md = f"> 💡 **Pro tip:** {b['tip']}"

    return dest_md, itinerary, budget_md, tip_md

custom_css = """
/* ── Page background ── */
body, .gradio-container {
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364) !important;
    font-family: 'Segoe UI', sans-serif !important;
}

/* ── Main container ── */
.main-wrap { max-width: 1100px; margin: auto; }

/* ── Hero banner ── */
.hero-box {
    background: linear-gradient(135deg, #f093fb 0%, #f5576c 50%, #fda085 100%);
    border-radius: 20px;
    padding: 2.5rem 2rem 2rem;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0 8px 32px rgba(240,147,251,0.35);
}
.hero-box h1 { font-size: 2.8rem; margin: 0; color: white; text-shadow: 0 2px 8px rgba(0,0,0,0.3); }
.hero-box p  { font-size: 1.1rem; color: rgba(255,255,255,0.92); margin: 0.5rem 0 0; }

/* ── Input panel ── */
.input-panel {
    background: rgba(255,255,255,0.07);
    border: 1px solid rgba(255,255,255,0.15);
    border-radius: 16px;
    padding: 1.5rem;
    backdrop-filter: blur(10px);
}

/* ── Labels ── */
.gradio-container label span {
    color: #f0e6ff !important;
    font-weight: 600 !important;
    font-size: 0.95rem !important;
}

/* ── Textboxes & dropdowns ── */
.gradio-container input[type=text],
.gradio-container textarea,
.gradio-container select {
    background: rgba(255,255,255,0.12) !important;
    border: 1px solid rgba(255,255,255,0.25) !important;
    border-radius: 10px !important;
    color: white !important;
    font-size: 1rem !important;
}
.gradio-container input[type=text]::placeholder,
.gradio-container textarea::placeholder { color: rgba(255,255,255,0.45) !important; }

/* ── Radio buttons ── */
.gradio-container .wrap.svelte-1viwdyb { color: white !important; }

/* ── Slider ── */
.gradio-container input[type=range] { accent-color: #f5576c; }

/* ── Generate button ── */
.gen-btn button {
    background: linear-gradient(90deg, #f093fb, #f5576c, #fda085) !important;
    border: none !important;
    border-radius: 50px !important;
    color: white !important;
    font-size: 1.2rem !important;
    font-weight: 700 !important;
    padding: 0.85rem 2rem !important;
    box-shadow: 0 4px 20px rgba(245,87,108,0.5) !important;
    transition: transform 0.2s, box-shadow 0.2s !important;
    letter-spacing: 1px;
}
.gen-btn button:hover {
    transform: scale(1.03) !important;
    box-shadow: 0 6px 28px rgba(245,87,108,0.7) !important;
}

/* ── Output panels ── */
.output-box {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
    border-radius: 16px !important;
    color: #f0f0f0 !important;
    padding: 1.2rem !important;
}
.output-box .prose, .output-box p,
.output-box h1, .output-box h2, .output-box h3,
.output-box li, .output-box td, .output-box th {
    color: #f0f0f0 !important;
}
.output-box table { border-collapse: collapse; width: 100%; }
.output-box td, .output-box th {
    border: 1px solid rgba(255,255,255,0.2) !important;
    padding: 8px 12px;
}
.output-box th { background: rgba(240,147,251,0.2) !important; }

/* ── Section dividers ── */
.divider { border: none; border-top: 1px solid rgba(255,255,255,0.12); margin: 1rem 0; }

/* ── Accordion ── */
.gradio-container .gradio-accordion {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
    border-radius: 12px !important;
}

/* ── Confetti canvas ── */
#confetti-canvas {
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    pointer-events: none;
    z-index: 9999;
}

/* ── Happy travels banner ── */
@keyframes slideDown {
    0%   { transform: translateY(-100px) translateX(-50%); opacity: 0; }
    20%  { transform: translateY(0)      translateX(-50%); opacity: 1; }
    80%  { transform: translateY(0)      translateX(-50%); opacity: 1; }
    100% { transform: translateY(-100px) translateX(-50%); opacity: 0; }
}
@keyframes pulse {
    0%, 100% { transform: translateX(-50%) scale(1);    }
    50%       { transform: translateX(-50%) scale(1.06); }
}
#happy-banner {
    display: none;
    position: fixed;
    top: 24px;
    left: 50%;
    transform: translateX(-50%);
    background: linear-gradient(90deg, #f093fb, #f5576c, #fda085);
    color: white;
    font-size: 1.4rem;
    font-weight: 700;
    padding: 1rem 2.5rem;
    border-radius: 50px;
    box-shadow: 0 8px 32px rgba(245,87,108,0.55);
    z-index: 10000;
    white-space: nowrap;
    letter-spacing: 1px;
}
"""

# ── Build the UI ───────────────────────────────────────────
with gr.Blocks(css=custom_css, title="✈️ AI Trip Planner") as demo:

    # Hero banner
    gr.HTML("""
    <div class="hero-box">
        <h1>🏞️ AI Trip Planner 🏞️ </h1>
        <p>Personalized itineraries powered by LLaMA 3.3 · Budgets in Indian Rupees · Made for  travelers</p>
    </div>
    """)

    with gr.Row():
        # ── Left: Inputs ──────────────────────────────────
        with gr.Column(scale=4, elem_classes="input-panel"):
            gr.HTML("<h3 style='color:#f5c6ff;margin:0 0 1rem'>🗺️ Plan your trip</h3>")

            city = gr.Textbox(
                label="📍 Where do you want to go?",
                placeholder="e.g. Tokyo  or  Bali  or  Jaipur (one city only!)",
                lines=1
            )
            interests = gr.Textbox(
                label="🎯 What are you into?",
                placeholder="street food, temples, hiking, nightlife, art...",
                lines=2
            )
            travel_style = gr.Radio(
                label="🎭 Your travel personality",
                choices=["🏛️ Culture Buff","⚡ Adventure Seeker","🍜 Foodie Explorer","💎 Luxury Traveler","📸 Insta Hunter"],
                value="🍜 Foodie Explorer"
            )
            with gr.Row():
                days = gr.Slider(label="📅 Trip duration (days)", minimum=1, maximum=14, value=3, step=1)
                budget_level = gr.Radio(
                    label="💸 Budget tier",
                    choices=["Budget","Mid-range","Luxury"],
                    value="Mid-range"
                )

            gr.HTML("<br>")
            generate_btn = gr.Button("🚀 Generate My Trip!", elem_classes="gen-btn", variant="primary")

        # ── Right: Outputs ────────────────────────────────
        with gr.Column(scale=6):

            with gr.Accordion("📌 Trip Summary", open=True, elem_classes="output-box"):
                dest_output = gr.Markdown(
                    value="*Your trip summary will appear here...*",
                    elem_classes="output-box"
                )

            gr.HTML("<div class='divider'></div>")

            with gr.Accordion("🗓️ Full Day-by-Day Itinerary", open=True, elem_classes="output-box"):
                itinerary_output = gr.Markdown(
                    value="*Your personalized itinerary will appear here after you click Generate!* ✈️",
                    elem_classes="output-box"
                )

            gr.HTML("<div class='divider'></div>")

            with gr.Accordion("💰 Budget Breakdown (₹ INR)", open=True, elem_classes="output-box"):
                budget_output = gr.Markdown(
                    value="*Budget details will appear here...*",
                    elem_classes="output-box"
                )

            pro_tip = gr.Markdown("", elem_classes="output-box")

    # ── Footer ────────────────────────────────────────────



    # ── Wire up ───────────────────────────────────────────
    generate_btn.click(
        fn=trip_planner,
        inputs=[city, interests, travel_style, days, budget_level],
        outputs=[dest_output, itinerary_output, budget_output, pro_tip],
        show_progress="full"
    )

# ── Launch ────────────────────────────────────────────────
print("\n" + "🌍"*20)
print("  LAUNCHING ENHANCED TRAVEL PLANNER")

demo.launch(share=True, debug=False)

In [ ]:
# ============================================
# PUSH YOUR TRAVEL PLANNER TO GITHUB
# ============================================

import os

# !!! CHANGE THESE 2 LINES (put them in quotes as strings) !!!
GITHUB_USERNAME = "bhumikasingh11"  # Your GitHub username in quotes
GITHUB_EMAIL = "bhumikasingh05dec@gmail.com"  # Your GitHub email in quotes
REPO_NAME = "travel-planner-ai"

# Configure git
os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
os.system(f'git config --global user.email "{GITHUB_EMAIL}"')

# Remove old git if exists
os.system("rm -rf .git")

# Initialize git
os.system("git init")
os.system("git add .")
os.system('git commit -m "Initial commit: AI Travel Planner with budget in INR"')

# Add remote and push
os.system(f'git remote add origin https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git')
os.system("git branch -M main")
os.system("git push -u origin main --force")

print("\n" + "="*50)
print("✅ SUCCESS! Check your GitHub:")
print(f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")
print("="*50)

In [ ]:
from google.colab import files
files.download('app.py')
files.download('requirements.txt')

In [ ]:
%%writefile app.py
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

# Get API key from environment
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

llm = ChatGroq(temperature=0, groq_api_key=GROQ_API_KEY, model_name="llama-3.3-70b-versatile")

def calculate_budget_inr(city, budget_level="Mid-range"):
    ex = 86.50
    if budget_level == "Budget":
        daily = 5000
    elif budget_level == "Luxury":
        daily = 20000
    else:
        daily = 12000
    return f"💰 Daily Budget: ₹{daily:,} | Total for 3 days: ₹{daily * 3:,}"

def generate_itinerary(city, interests, days):
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"Create a {days}-day itinerary for {city}. Interests: {interests}. Use emojis and bullet points."),
        ("human", "Create itinerary")
    ])
    response = llm.invoke(prompt.format_messages())
    return response.content

with gr.Blocks(title="AI Travel Planner") as demo:
    gr.Markdown("# ✈️ AI Travel Planner\n### Personalized itineraries with budget in Indian Rupees")

    with gr.Row():
        with gr.Column():
            city = gr.Textbox(label="🏙️ City", placeholder="Paris, Tokyo, Bali...")
            interests = gr.Textbox(label="🎯 Interests", placeholder="museums, food, nature", lines=2)
            days = gr.Slider(label="📅 Days", minimum=1, maximum=14, value=3)
            budget_level = gr.Radio(label="💰 Budget", choices=["Budget", "Mid-range", "Luxury"], value="Mid-range")
            generate_btn = gr.Button("🚀 Generate")

        with gr.Column():
            itinerary_out = gr.Markdown(label="🗺️ Itinerary")
            budget_out = gr.Markdown(label="💰 Budget")

    generate_btn.click(fn=generate_itinerary, inputs=[city, interests, days], outputs=itinerary_out)
    budget_btn = gr.Button("💰 Show Budget")
    budget_btn.click(fn=calculate_budget_inr, inputs=[city, budget_level], outputs=budget_out)

print("✅ app.py created")

In [ ]:
%%writefile requirements.txt
gradio>=5.0.0
langchain>=0.3.0
langchain-groq>=0.2.0
langchain-core>=0.3.0

In [ ]:
%%writefile README.md
# ✈️ AI Travel Planner

[![Python](https://img.shields.io/badge/Python-3.10-blue)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3-green)](https://langchain.com)
[![Gradio](https://img.shields.io/badge/Gradio-5.0-orange)](https://gradio.app)

## 🎯 Features
- AI-generated personalized itineraries
- Budget calculation in Indian Rupees (₹)
- 5 travel styles support
- Downloadable trip plans

## 🛠️ Tech Stack
- LangChain + Groq API (LLaMA 3.3 70B)
- Gradio UI
- Python

## 🔧 Setup
```bash
git clone https://github.com/bhumikasingh11/travel-planner-ai
cd travel-planner-ai
pip install -r requirements.txt
export GROQ_API_KEY="your_key"
python app.py


### **Cell 4: Create .gitignore**
```python
%%writefile .gitignore
__pycache__/
*.pyc
.DS_Store
.env